In [70]:
import openai
from typing import List

client = openai.OpenAI() # OpenAI API에 요청을 보내고 응답받는 기능을 제공하는 객체
response = client.chat.completions.create( # OpenAI의 Chat API를 사용하여 대화를 생성
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕하세요!"}]
)
response.choices[0].message.content

'안녕하세요! 어떻게 도와드릴까요?'

In [71]:
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요."
def call_chat_model(messages: List[dict]):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
    )
    return response.choices[0].message.content

def invoke_chain(topic: str):
    prompt_value = prompt_template.format(topic=topic)
    messages = [{'role':'user','content':prompt_value}]
    return call_chat_model(messages)
invoke_chain("더블딥")

"더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체 또는 recession이 발생한 후 일정 기간의 회복이 있었지만 다시 경기 후퇴로 접어드는 상황을 설명합니다. 즉, 경제가 한 번 반등한 뒤 다시 침체되는 두 개의 '딥'을 경험하는 것을 의미합니다. 이러한 현상은 소비자 신뢰와 기업 투자에 부정적인 영향을 미치며, 지속적인 경제 회복을 어렵게 만드는 요소로 작용할 수 있습니다."

In [72]:
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")
llm = OpenAI(api_key=api_key)
llm.invoke("안녕")

'하세요. 저는 김동규입니다.\n저는 1991년 5월 2일에 태어났고, 현재는 서울에서 살고 있습니다. 저는 IT 관련 회사에서 일하고 있으며, 주로 웹 개발 분야에서 활동하고 있습니다.\n\n저는 어릴 적부터 컴퓨터에 관심이 많아서 프로그래밍을 시작하게 되었습니다. 그래서 컴퓨터 공학을 전공하고 졸업 후에도 계속하여 개발 분야에서 일하고 있습니다.\n\n저는 항상 새로운 기술과 도전을 좋아하며, 문제를 해결하는 것에 큰 즐거움을 느끼고 있습니다. 또한 협업을 중요하게 생각하며, 다른 사람들과 함께'

In [73]:
from langchain_openai import ChatOpenAI #랭체인에서 OpenAI의 GPT 모델을 사용하는 클래스
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿을 만들어주는 클래스
from langchain_core.output_parsers import StrOutputParser # GPT모델이 반환한 응답을 문자열로 반환, 모델의 응답은 다양한 형태로 반환될 수 있는데, 그 중에서 문자열 응답만 추출하여 반환
from langchain_core.runnables import RunnablePassthrough # 입력 데이터를 그대로 통과시키는 역할
from dotenv import load_dotenv

prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")
output_parser = StrOutputParser() # 출력 파서를 문자열로 설정
model = ChatOpenAI(model='gpt-4o')
# 파이프라인 설정: 주제를 받아 프롬프트를 생성하고, 모델로 응답을 생성한 후 문자열로 파싱
chain = (
    {"topic":RunnablePassthrough()} # 입력받은 주제를 그대로 통과
    | prompt # 프롬프트 템플릿 적용
    | model #모델을 사용하여 응답 생성
    | StrOutputParser() # 응답을 문자열로 파싱
)
chain.invoke("더블딥")

'"더블딥"은 일반적으로 경제 또는 금융 분야에서 사용되는 용어로, 경제가 침체에 빠진 후 회복되지 않고 다시 하락하는 현상을 의미합니다. 첫 번째 하락 이후 일시적으로 회복되는 듯하다가 다시 침체가 오는 모양이 마치 "W"자 형태와 유사하다고 하여 이렇게 불립니다. 이러한 상황은 다양한 요인, 예를 들어 정책 실패, 외부 충격, 소비자 및 기업 신뢰 부족 등으로 인해 발생할 수 있습니다. 더블딥 경기 침체는 경제에 장기적인 영향을 미칠 수 있으므로 주의 깊게 관찰하고 대응할 필요가 있습니다.'

In [74]:
model = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")
parser = StrOutputParser()
chain = prompt | model | parser # 프롬프트, 모델, 출력 파서를 체인으로 연결
chain.invoke({"topic":"더블딥"}) # 단일 주제
chain.batch([{"topic":"더블딥"}, {"topic":"인플레이션"}]) # 여러 주제를 동시에 처리
for token in chain.stream({"topic":"더블딥"}): #응답을 토큰 단위로 스트리밍하여 출력
    print(token, end="", flush=True) # 스트리밍된 내용을 출력, 각 내용을 붙여서 출력하며 버퍼를 즉시 플러시하여 실시간으로 보여줌

더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기침체가 끝난 후 잠시 회복을 보이다가 다시 경기침체로 돌아가는 현상을 설명합니다. 즉, 경제가 두 번의 침체를 겪는 상황을 의미합니다. 이 경우 첫 번째 침체 이후에 발생하는 회복세가 일시적이며, 여러 가지 이유로 인해 다시 하강세로 전환될 수 있습니다. 더블딥은 경제정책 결정자들에게 큰 도전 과제가 되며, 기업과 개인에게도 불확실성을 증가시킵니다.

In [75]:
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해줘  : {answer}")
# 순서 : "answer"의 키에 chain에서 생성된 응답이 들어가고, 이 응답을 영어로 번역하는 프롬프트로 전달, 이 후 모델을 실행하여 번역된 응답을 생성하고, 결과를 StrOutputParser를 통해 문자열로 변환
composed_chain = {"answer":chain} | analysis_prompt | model | parser
composed_chain.invoke({"topic":"더블딥"})

'"Double Dip" refers to a situation in economics where, after a recession, the economy does not recover but instead falls back into recession. In other words, it shows a pattern where the economy seems to recover temporarily but then declines again. This phenomenon can occur as indicators such as consumer confidence, business investment, and employment continue to deteriorate, and it may be triggered by insufficient effectiveness of economic policies or external shocks. Double dips significantly impact economic uncertainty and the sentiments of consumers and investors, prompting ongoing efforts to address the situation through monetary or fiscal policies.'

In [76]:
# 함수를 러너블로 강제 변환
model = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")
parser = StrOutputParser()
chain = prompt | model | parser
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해줘  : {answer}")

composed_chain_with_lambda=(
    chain # 정의한 chain을 사용하여 입력된 데이터에 대한 설명을 받음
    | (lambda input: {"answer":input}) #설명을 "answer"의 key로 변환
    | analysis_prompt # 'answer'의 key를 영어로 번역하도록 하는 프롬프트에 전달
    | model # 모델이 번역한 결과 생성
    | StrOutputParser() # 문자열로 파싱
)

composed_chain_with_lambda.invoke({"topic":"더블딥"})

'The term "Double Dip" refers to a phenomenon in economics where a recession occurs twice in succession. It describes a situation where the economy appears to have temporarily recovered after the first recession, only to fall back into a recession again. This situation can lead to a decrease in consumption and investment, a rise in unemployment, and can have negative effects on the economy as a whole. Double dips are particularly concerning when there is significant uncertainty regarding economic recovery.'

In [77]:
# __or__을 사용한 파이썬 오버로딩
class CustomLCEL:
    def __init__(self, value):
        self.value = value

    def __or__(self,other):
        if callable(other): # other가 함수일 경우, 함수를 호출하고, 그 결과를 새로운 객체로 반환
            return CustomLCEL(other(self.value))
        else: #other가 함수가 아니면 오류
            raise ValueError("Right operand must be callable")
    def result(self):
        return self.value # 현재 값 반환

def add_exclamation(s): #문자열 끝 느낌표 추가
    return s + "!"
def reverse_string(s): #문자열 뒤집기
    return s[::-1]

custom_chain=(
    CustomLCEL("랭체인 공부하기")
    | add_exclamation
    | reverse_string
)

result = custom_chain.result()
print(result)

!기하부공 인체랭


In [78]:
# 파이프 메소드 방법 1
composed_chain_with_pipe =(
    chain
    .pipe(lambda input: {"answer":input})
    .pipe(analysis_prompt)
    .pipe(model)
    .pipe(StrOutputParser())
)

composed_chain_with_pipe.invoke({'topic':"더블딥"})


#방법 2
composed_chain_with_pipe = chain.pipe(lambda input:{"answer":input}, analysis_prompt, model, StrOutputParser())
composed_chain_with_pipe.invoke({"topic":"더블딥"})


'The term "Double Dip" is primarily used in economics to refer to a situation where an economy experiences a recession, temporarily recovers, and then falls back into another recession. This phenomenon occurs instead of a V-shaped recovery, appearing as a U-shaped or L-shaped recovery, where corporate investment and consumer confidence decline again, leading to a slowdown in economic activity. It can arise due to financial crises or other economic factors and can pose challenges for policymakers.'

In [79]:
# RunnableParallel을 통한 체인 구성
from langchain_core.runnables import RunnableParallel

model = ChatOpenAI()
kor_chain=(
    ChatPromptTemplate.from_template("{topic}에 대해서 짧게 설명해줘.")
    | model
    | StrOutputParser()
)

eng_chain=(
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명해줘.")
    | model
    | StrOutputParser()
)

# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
result = parallel_chain.invoke({"topic":"더블딥"})

print("한글 설명: ",result['kor'])
print("영어 설명: ",result['eng'])

한글 설명:  더블딥(Double Deep)은 딥러닝 모델 중 하나로, 인공 신경망이 깊게 쌓여있는 모델을 의미합니다. 더블딥은 보다 더 복잡한 문제들을 해결하기 위해 더 깊은 층을 가진 모델을 사용하며, 성능이 뛰어난 경우가 많습니다. 그러나 모델이 깊어질수록 학습이 어려워지고 과적합이 발생할 수 있으므로 주의가 필요합니다.
영어 설명:  Double dip refers to a situation where someone benefits or profits twice from the same action or investment. It can also refer to a strategy in which someone takes advantage of two separate but related opportunities to gain an extra advantage.


In [80]:
# 문자열 프롬프트 템플릿
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("주제 {topic}에 대해서 금융과 관련된 간략한 조언을 해줘.")
prompt_template.invoke({"topic":"투자"})

StringPromptValue(text='주제 투자에 대해서 금융과 관련된 간략한 조언을 해줘.')

In [81]:
# 챗 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([ #ChatPromptTemplate: 사용자와 시스템 간의 메시지 포함
    ("system", "당신은 유능한 금융 조언가입니다."), # 시스템 메시지: AI의 역할을 정의, AI가 어떤 종류의 응답을 제공해야 하는지 정의
    ("user", "주제 {topic}에 대해 금융 관련 조언을 해주세요.") # 사용자 메시지: 사용자가 요청하는 내용을 포함하여, AI에게 특정 정보를 요청
])
prompt_template.invoke({"topic":"주식"})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='주제 주식에 대해 금융 관련 조언을 해주세요.', additional_kwargs={}, response_metadata={})])

In [82]:
# 매사자 자리 표시자
from langchain_core.prompts import ChatMessagePromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

# 방법 1: 메시지 자리 표시자를 포함한 ChatPromptTemplate 설정
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 금융 조언가입니다."),
    MessagesPlaceholder("msgs")
])
prompt_template.invoke({"msgs":[HumanMessage(content="안녕하세요!")]}) # 메시지 리스트를 'msgs' 자리 표시자에 전달하여 호출


ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [83]:
# 방법 2: MessagePlaceholder 클래스를 사용하지 않고 비슷한 작업 수행
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 금융 조언가입니다."),
    ("placeholder","{msgs}") # msgs가 자리 표시자로 사용
])

prompt_template.invoke({"msgs":[HumanMessage(content='안녕하세요!')]})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [84]:
# 퓨삿 예제 프롬프트 템플릿
example_prompt = PromptTemplate.from_template("질문: {question}\n답변: {answer}")
examples = [{
    "question":"주식 투자와 예금 중 어느 것이 더 수익률이 높은가?",
    "answer" : """
    후속 질문이 필요하신가요: 네.
    후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
    중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
    후속 질문: 예금의 평균 이자율은 얼마인가요?
    중간 답변: 예금의 평균 이자율은 연 1%입니다.
    따라서 최종 답변은: 주식투자
    """,
},
{
    "question":"부동산과 채권 중 어느 것이 더 안정적인 투자처인가?",
    "answer": """
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    """,
},
]

print(example_prompt.invoke(examples[0]).to_string())

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
    후속 질문이 필요하신가요: 네.
    후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
    중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
    후속 질문: 예금의 평균 이자율은 얼마인가요?
    중간 답변: 예금의 평균 이자율은 연 1%입니다.
    따라서 최종 답변은: 주식투자
    


In [85]:
# 퓨샷 프롬프트 템플릿
from langchain_core.prompts import FewShotPromptTemplate
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="질문: {input}",
    input_variables=['input'],
)

print(prompt.invoke({"input": "부동산 투자의 장점은 무엇인가?"}).to_string()
)

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
    후속 질문이 필요하신가요: 네.
    후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
    중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
    후속 질문: 예금의 평균 이자율은 얼마인가요?
    중간 답변: 예금의 평균 이자율은 연 1%입니다.
    따라서 최종 답변은: 주식투자
    

질문: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
답변: 
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    

질문: 부동산 투자의 장점은 무엇인가?


In [86]:
# 예제 선택기 
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, # 사용할 예제 목록
    OpenAIEmbeddings(api_key=api_key), # 임베딩 생성에 사용하는 클래스
    Chroma, # 임베딩을 저장하고 유사도 검색을 수행하는 벡터 저장소 클래스
    k=1, # 선택할 예제 갯수
)
question = "부동산 투자의 장점은 무엇인가?"
selected_examples = example_selector.select_examples({"question":question})
print(f"입력 질문:{question}")
for example in selected_examples:
    print("\n")
    print("# 입력과 가장 유사한 예제:")
    for k, v in example.items():
        print(f"{k}:{v}")

입력 질문:부동산 투자의 장점은 무엇인가?


# 입력과 가장 유사한 예제:
question:부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
answer:
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    


In [87]:
# 퓨샷 프롬프트 + 실제 모델과 함께 사용하는 예시 코드
example_prompt = PromptTemplate(
    input_variables=["question",'answer'], # 해당 템플릿은 2개의 입력 변수를 사용
    template='질문: {question}\n답변: {answer}' # 실제로 질문과 답변을 표시하는 형식
)

prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt = example_prompt, # 앞서 정의한 질문과 답변 형식의 프롬프트 템플릿을 사용하여 예제를 제공할 수 있도록 함
    prefix="다음은 금융 관련 질문과 답변의 예입니다:", # 프롬프트 앞부분에 붙는 텍스트
    suffix="질문: {input}\n답변:", # 질문 후 모델이 답변을 생성해야 할 부분
    input_variables=['input'] # 실제로 사용자가 입력한 질문
)

model = ChatOpenAI(model_name='gpt-4o')
chain = prompt | model
response = chain.invoke({"input":"부동산 투자의 장점은 무엇인가?"})
print(response)
print(response.content)

content='부동산 투자의 장점으로는 몇 가지가 있습니다:\n\n1. **안정적인 수익**: 부동산은 임대 수익을 통해 안정적인 현금 흐름을 제공합니다. 특히 주거용 부동산은 비교적 꾸준한 임대 수익을 기대할 수 있습니다.\n\n2. **가치 상승 가능성**: 장기적으로 부동산 가격은 상승하는 경향이 있어, 자산 가치 상승을 기대할 수 있습니다. 특히 개발 지역이나 인프라 개선이 이루어지는 지역에서는 그 가치가 더 크게 상승할 가능성이 있습니다.\n\n3. **인플레이션 헤지**: 부동산은 인플레이션에 대해 강한 방어력을 갖고 있습니다. 물가 상승에 따라 임대료나 자산의 가치도 상승하게 됩니다.\n\n4. **세제 혜택**: 부동산에는 다양한 세제 혜택이 존재할 수 있습니다. 예를 들어, 모기지 이자나 감가상각비를 일부 공제받을 수 있으며, 양도소득세 면제 대상이 될 수도 있습니다.\n\n5. **자산의 실제성**: 부동산은 눈에 보이고, 물리적으로 존재하는 자산으로서 심리적 안정감을 주기도 합니다.\n\n각 투자자는 자신의 투자 목표와 리스크 허용 범위를 고려하여 부동산 투자가 적절한지 판단해야 합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 302, 'prompt_tokens': 152, 'total_tokens': 454, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerpri

In [88]:
formatted_prompt = prompt.invoke({
    "input": "부동산 투자의 장점은 무엇인가?"
}).to_string()

print(formatted_prompt)

다음은 금융 관련 질문과 답변의 예입니다:

질문: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
답변: 
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    

질문: 부동산 투자의 장점은 무엇인가?
답변:


In [89]:
'''
from langchain_classic import hub 
prompt = hub.pull("hardkothari/prompt-maker") 
hub.pull("hardkothari/prompt-maker:c5db8eee")
'''
from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "hardkothari/prompt-maker",
    dangerously_pull_public_prompt=True
)
client.pull_prompt(
    "hardkothari/prompt-maker:c5db8eee",
    dangerously_pull_public_prompt=True
)

ChatPromptTemplate(input_variables=['lazy_prompt', 'task'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hardkothari', 'lc_hub_repo': 'prompt-maker', 'lc_hub_commit_hash': 'c5db8eeefa7be4862a9599b759608dd10ee53f53910838f69abb5ab31c257c2d'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert Prompt Writer for Large Language Models.\n\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['lazy_prompt', 'task'], input_types={}, partial_variables={}, template='Your goal is to improve the prompt given below for {task} :\n--------------------\n\nPrompt: {lazy_prompt}\n\n--------------------\n\nHere are several tips on writing great prompts:\n\n-------\n\nStart the prompt by stating that it is an expert in the subject.\n\nPut instructions at the beginning of the prompt and use ### or to separate the instruction and context \n\nBe specific, 

In [90]:
from langchain_core.output_parsers import JsonOutputParser
parser=JsonOutputParser()
instructions = parser.get_format_instructions() # 출력파서의 포맷 지침 확인
print(instructions)


Return a JSON object.


In [91]:
ai_response = '{"이름":"김철수","나이":30}'
parsed_response = parser.parse(ai_response)
print(parsed_response)


{'이름': '김철수', '나이': 30}


In [92]:
from langchain_classic.output_parsers import RetryWithErrorOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import OpenAI

parser=RetryWithErrorOutputParser.from_llm(parser=JsonOutputParser(),llm=ChatOpenAI())
question = "가장 큰 대륙은?"
ai_response = "아시아입니다."  # JSON 형식이 아닌 잘못된 응답
try:
    result = parser.parse_with_prompt(ai_response, question)
    print(result)
except Exception as e:
    print(f"오류 발생: {e}")
    # 여기서 AI에게 다시 질문할 수 있음

오류 발생: 'str' object has no attribute 'to_string'


In [93]:
from langchain_core.output_parsers import PydanticOutputParser # 모델의 출력을 Pydantic 모델에 맞게 구조화된 데이터로 변환, 일관된 형식과 데이터 검증 제공
from langchain_core.prompts import PromptTemplate # 프롬프트를 동적으로 생성하여 모델에게 전달할 질문과 출력 형식 지침을 설정, 변수를 포함하여 다양한 입력을 처리할 수 있음
from langchain_openai import OpenAI
from pydantic import BaseModel, Field, model_validator # Python에서 데이터 검증과 모델링을 위한 라이브러리(Pyndantic), BaseModel과 Field로 데이터 구조를 정의, model_validator로 입력데이터 검증

model = ChatOpenAI(model='gpt-4o', temperature=0.0)

# 원하는 데이터 구조 정의
class FinancialAdvice(BaseModel):
    setup: str = Field(description="금융 조언 상황을 설정하기 위한 질문")
    advice: str = Field(description='질문을 해결하기 위한 금융 답변')
    # Pydantic을 사용한 사용자 정의 검증 로직
    @model_validator(mode="before") # Pydantic 모델 전체를 검증하는 함수로 등록하는 데코레이터, 질문 형식이 올바른지 검증하는 로직, 사용자의 Query를 검사하는 것이 아닌, 모델이 만든 출력을 검증
    @classmethod # question_ends_with_question_mark 메소드가 인스턴스 메소드가 아니라 클래스 메소드(그래서 self를 안쓰고, cls를 씀)
    def question_ends_with_question_mark(cls, values:dict) -> dict: # 흐름도 : @classmethod → @model_validator, 해당 메소드가 Pydantic 검증기로 등록되면서 FinancialAdvice 클래스 전체의 검증 과정에 영향을 줌
        setup = values.get("setup","")
        if not setup.endswith("?"):
            raise ValueError("잘못된 질문 형식입니다! 질문은 '?'로 끝나야 합니다.")
        return values

parser = PydanticOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate(
    template='다음 금융 관련 질문에 답변해줘.\n{format_instructions}\n질문:{query}\n',
    input_variables=["query"],
    partial_variables ={"format_instructions":parser.get_format_instructions()}, # 매번 입력받을 필요 없는 고정 프롬프트 변수들을 미리 채워두는 역할
)
chain = prompt | model | parser

try:
    result = chain.invoke({"query":"부동산에 관련하여 금융 조언을 받을 수 있게 질문해."})
    print(result)
except Exception as e:
    print(f"오류 발생: {e}")


오류 발생: Failed to parse FinancialAdvice from completion {"setup": "부동산 투자에 관심이 있는 사람입니다. 현재 시장 상황과 투자 전략에 대한 조언을 받고 싶습니다.", "advice": "부동산 투자를 고려할 때는 먼저 시장 조사를 철저히 하는 것이 중요합니다. 지역의 경제 성장률, 인구 증가율, 인프라 개발 계획 등을 분석하여 미래 가치 상승 가능성이 높은 지역을 선택하세요. 또한, 투자 목적에 따라 주거용, 상업용, 또는 토지 투자 중 어떤 것이 적합한지 결정해야 합니다. 금융적으로는 대출 금리, 세금 혜택, 그리고 예상 수익률을 고려하여 투자 규모와 방식을 계획하세요. 전문가의 조언을 받거나 부동산 투자 펀드를 통해 간접 투자를 고려하는 것도 좋은 방법입니다."}. Got: 1 validation error for FinancialAdvice
  Value error, 잘못된 질문 형식입니다! 질문은 '?'로 끝나야 합니다. [type=value_error, input_value={'setup': '부동산 투...은 방법입니다.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [94]:
# SimpleJsonOutputParser
from langchain_core.output_parsers.json import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(
    "다음 질문에 대한 답변이 포함된 JSON 객체를 반환하세요: {question}"
)
json_parser = SimpleJsonOutputParser()
json_chain = json_prompt | model | json_parser
# 스트리밍 예시
list(json_chain.stream({"question": "비트코인에 대한 짧은 한 문장 설명."}))

[{},
 {'description': ''},
 {'description': '비'},
 {'description': '비트'},
 {'description': '비트코'},
 {'description': '비트코인은'},
 {'description': '비트코인은 분'},
 {'description': '비트코인은 분산'},
 {'description': '비트코인은 분산된'},
 {'description': '비트코인은 분산된 디'},
 {'description': '비트코인은 분산된 디지털'},
 {'description': '비트코인은 분산된 디지털 화'},
 {'description': '비트코인은 분산된 디지털 화폐'},
 {'description': '비트코인은 분산된 디지털 화폐로'},
 {'description': '비트코인은 분산된 디지털 화폐로,'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인 간'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인 간의'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인 간의 거래'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인 간의 거래를'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인 간의 거래를 가능'},
 {'description': '비트코인은 분산된 디지털 화폐로, 중앙 기관 없이 개인 간의 거래를 가능하게'},
 {'description': '비트

In [95]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

model = ChatOpenAI(temperature=0.0)
# 원하는 데이터 구조 정의
class FinancialAdvice(BaseModel):
    setup:str = Field(description="금융 조언 상황을 설정하기 위한 질문")
    advice:str = Field(description='질문을 해결하기 위한 금융 답변')

# Json 출력 파서 설정 및 프롬프트 템플릿에 지침 삽입
parser = JsonOutputParser(pydantic_object = FinancialAdvice)
prompt = PromptTemplate(
    template="다음 금융 관련 질문에 답변해주세요.\n{format_instructions}\n{query}\n",
    input_variables=['query'],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | model | parser
chain.invoke({"query":"부동산에 관련하여 금융 조언을 받을 수 있게 질문해!"})


{'setup': '부동산 투자를 시작하려면 어떤 절차를 거쳐야 하나요?',
 'advice': '부동산 투자를 시작하기 전에 자신의 재정 상황을 정확히 파악하고 목표를 설정하는 것이 중요합니다. 또한 지역별 부동산 시장 동향과 투자 방법에 대해 충분한 조사를 하고 전문가의 조언을 듣는 것이 좋습니다.'}

In [96]:
# 기본적인 대화 이력 전달 : 이전 대화를 그대로 프롬프트에 전달
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        ("placeholder", "{messages}"), # 대화 이력 추가
    ]
)

chain = prompt | chat
ai_msg = chain.invoke( # 이전 대화를 포함한 메세지 전달
    {
        "messages":[
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"), # 사용자의 첫 질문
            ("ai", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."), # 챗봇의 답변
            ("human", "방금 뭐라고 했나요?"), # 사용자의 재확인 질문
        ],
    }
)
print(ai_msg.content)

저축을 늘리기 위해 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하는 방법을 추천했습니다. 이렇게 하면 꾸준하게 저축할 수 있으며, 목표 달성에 도움이 됩니다. 추가적인 조언이 필요하시면 말씀해 주세요!


In [97]:
from langchain_community.chat_message_histories import ChatMessageHistory
chat_history = ChatMessageHistory() # 대화 이력 저장을 위한 클래스 초기화
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)


저축을 늘리기 위해 저축 목표를 설정하고 매달 자동 이체로 일정 금액을 저축하는 방법을 추천했습니다. 이렇게 하면 저축을 습관화할 수 있습니다. 추가로 예산을 세워 지출을 관리하거나, 불필요한 지출을 줄이는 것도 도움이 됩니다. 필요하신 구체적인 방법이나 조언이 있다면 말씀해 주세요!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 시스템 메새지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하세요."),
        ("placeholder", "{chat_history}"), # 이전 대화 이력
        ("human", "{input}"), # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = ChatMessageHistory() # 대화 이력을 저장하고 관리하는 객체, 사용자의 이전 질문과 모델의 답변을 체계적으로 기록
chain = prompt | chat  # 이 체인 자체는 대화 기록을 자동으로 저장하지 않고, 입력받은 chat_history, input으로 프롬프트를 만들고 모델 호출하는 체인

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌈 : 대화 이력을 저장하고, 필요할 때 불러 올 수 있음
chain_with_message_history = RunnableWithMessageHistory(
    chain, # 실제로 실행할 체인
    lambda session_id: chat_history, # 세션 ID에 따라 특정 세션의 대화 이력 추적
    input_messages_key = "input", # 입력된 질문을 처리하기 위한 키 설정
    history_messages_key = "chat_history", # 대화 이력을 처리하는데 사용되는 키 설정
)

# 질문 메세지 체인 실행
print(chain_with_message_history.invoke(
    {"input":"저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable":{"session_id":"unused"}},
).content)



c:\Users\Dohy\Desktop\dohy\RAG_master\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


저축을 늘리기 위해 할 수 있는 여러 가지 방법이 있습니다. 다음은 몇 가지 팁입니다:

1. **예산 수립**: 수입과 지출을 파악하고, 필요한 항목과 불필요한 항목을 구분하여 예산을 세우세요. 이렇게 하면 어디서 절약할 수 있는지 명확해집니다.

2. **자동 저축**: 급여가 들어오자마자 일정 금액을 저축 계좌로 자동 이체하는 시스템을 설정하세요. 이 방법은 자신도 모르게 저축을 늘릴 수 있는 효과적인 방법입니다.

3. **지출 줄이기**: 불필요한 소비를 줄이세요. 외식, 커피, 쇼핑 등을 줄이는 것이 좋습니다. 작은 금액의 절약도 모이면 큰 금액이 될 수 있습니다.

4. **할인 및 프로모션 활용**: 구매할 때 할인 쿠폰이나 적립 프로그램을 활용하세요. 이는 장기적으로 큰 금액을 절약할 수 있습니다.

5. **비상 자금 마련**: 예상치 못한 지출에 대비해 비상 자금을 마련하세요. 이를 통해 큰 지출이 생겨도 저축을 방해받지 않도록 할 수 있습니다.

6. **목표 설정**: 저축 목표를 설정하세요. 단기, 중기, 장기 목표를 세우고 그 목표를 달성하기 위한 계획을 세우면 동기 부여에 도움이 됩니다.

7. **부수입 창출**: 추가적인 소득원을 찾아보세요. 프리랜서 작업, 투자, 온라인 판매 등 다양한 방법으로 부수입을 늘릴 수 있습니다.

8. **정기적으로 점검**: 자신의 저축 현황을 정기적으로 점검하고, 필요에 따라 예산이나 저축 계획을 조정하세요.

위의 방법들을 통해 저축을 꾸준히 늘려나갈 수 있습니다. 각자의 상황에 맞는 방법을 선택하여 실행해 보세요!


In [99]:
chain_with_message_history.invoke(
    {"input":"내가 방금 뭐라고 했나요?"}, # 사용자의 질문
    {"configurable":{"session_id":"unused"}} # 세션 ID 설정
).content

'당신은 "저축을 늘리기 위해 무엇을 할 수 있나요?"라고 질문하셨습니다. 저축을 늘리기 위한 여러 가지 방법에 대해 안내해 드렸습니다. 더 궁금한 점이 있으시면 언제든지 말씀해 주세요!'

In [ ]:
# 메세지 트리밍
'''
사용자 입력
↓
RunnableWithMessageHistory가 기존 대화 기록 불러옴
↓
chain_with_trimming 실행
↓
RunnablePassthrough.assign(...)이 chat_history를 trim함
↓
prompt에 input + 잘린 chat_history 전달
↓
chat 모델 호출
↓
모델 답변 반환
↓
RunnableWithMessageHistory가 새 human/ai 메시지를 history에 저장
'''

from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

trimmer = trim_messages(strategy="last",max_tokens=2, token_counter=len) # 메세지 트리밍 유틸리티 설정, strategy : 가장 최근 메세지를 기준으로, max_tokens = 남길 메세지(token_count=len으로 잡았기 때문에) 수
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer) # itemgetter를 통해 입력 딕셔너리에서 이전에 저장된 대화 이력(chat_history)을 불러온 후, 트리밍하여 프롬프트와 모델에 전달
                                                                                  # RunnablePassthrough : 기존 입력을 그대로 통과시키는 러너블, assign() : 기존 입력 딕셔너리에 새 key를 추가하거나 기존 key를 덮어씀 
    | prompt
    | chat
)

chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id : chat_history,
    input_messages_key='input', # 현재 사용자 입력이 딕셔너리의 'input'의 키로 있음
    history_messages_key = 'chat_history', # 이전 대화 기록을 체인 입력의 'chat_history' key로 넣음
)

chain_with_trimmed_history.invoke(
    {'input':"저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"}, # 사용자의 질문
    {'configurable':{"session_id":"finance_session_1"}}
)

chain_with_trimmed_history.invoke(
    {"input":"내가 방금 뭐라고 했나요?"},
    {"configurable":{"session_id":"finance_session_1"}}
).content

'당신은 "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"라고 질문하셨습니다. 제가 이에 대한 조언을 드렸습니다. 추가로 궁금한 점이 있으면 언제든지 말씀해 주세요!'

In [101]:
def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0 :
        return False
    
    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder","{chat_history}"), # 이전 대화 이력
            ("user","이전 대화를 요약해주세요. 가능한 한 많은 세부 정보를 포함하세요."), # 요약 요청 메세지
        ]
    )
    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | chat
    summary_message = summarization_chain.invoke({"chat_history":stored_messages})

    chat_history.clear() # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message) # 요약된 메세지를 대화 이력에 추가

    return True

In [102]:
# 대화 요약을 처리하는 체인 
chain_with_summarization =(
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

In [103]:
# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input":"저에게 어떤 재정적 조언을 해주셨나요?"}, # 사용자의 질문
    {"configurable":{"session_id":"unused"}}).content)

저는 다음과 같은 재정적 조언을 드렸습니다:

1. **예산 수립**: 수입과 지출을 명확히 파악하여 예산을 세우고, 불필요한 지출을 줄이는 것이 중요합니다.

2. **저축을 늘리기 위한 방법**:
   - **자동 저축**: 급여의 일정 부분을 자동으로 저축 계좌로 이체하도록 설정하여 저축을 습관화하세요.
   - **지출 줄이기**: 외식, 커피, 고급 소비 등을 줄여 저축할 수 있는 금액을 늘리세요.
   - **할인 및 프로모션 활용**: 구매할 때 할인 쿠폰이나 적립 프로그램을 활용하세요.
   - **비상 자금 마련**: 예기치 못한 상황에 대비하기 위해 비상 자금을 마련하는 것이 중요합니다.
   - **목표 설정**: 저축 목표를 설정하고 달성하기 위한 구체적인 계획을 세우세요.

3. **5년 내에 집 구매를 위한 재정 계획**:
   - **저축 목표 설정**: 원하는 집의 가격과 다운 페이먼트를 고려하여 필요한 자금을 계산하세요.
   - **신용 점수 관리**: 주택 대출을 받을 때 유리한 조건을 위해 신용 점수를 확인하고 개선하세요.
   - **부채 감소**: 고이자율의 부채 상환을 우선적으로 처리하세요.
   - **재정 계획 점검**: 정기적으로 저축 상황과 계획을 점검해 필요한 경우 조정하세요.
   - **전문가 상담**: 필요한 경우 비재정적 상담과 지원을 위해 전문가와 상담하세요.

이러한 조언들은 재정적으로 안정성을 높이고 장기적인 목표를 달성하는 데 도움이 될 수 있습니다. 더 궁금한 점이나 구체적인 상황에 대해 질문이 있으시면 언제든지 말씀해 주세요!
